In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import matlab.engine

import sys
sys.path.append('../../wavelet/')
import wavelet_funcs as wf

In [4]:
eng = matlab.engine.start_matlab()

### Dummy data

In [5]:
dt = 0.5
Fs = 1/dt
nt = 4021
t = np.arange(nt)*dt
T = nt*dt

fsignal = 0.03
amp = 1
x = amp*np.cos(2*np.pi*fsignal*t)

P = amp**2 / 2
E = P*T
E_signal = np.sum(x**2) * dt

In [6]:
tbw = 30
res = wf.matlab_cwt(eng, x, Fs, boundary='reflect', time_bandwidth=20)

In [7]:
wt_energy = (res.wt_amp**2) * res.scale # Multiply by scale to convert L1 to L2 normalization. Not sure this is correct thing to do

### From Paul Adisson "Illustrated Wavelet Handbook":

Total energy can be recovered from the scalogram (`wt_energy`) by integrating over scale $s$ and time $t$:

$$
E = \frac{1}{C_g} \int_{-\infty}^{\infty}dt \int_0^{\infty}\frac{ds}{s^2} |T(s, t)|^2,
$$

where $C_g$ is the admissibility constant.

In [10]:
Cg = wf.admissibility_constant_gmw_matlab(timebandwidth=tbw)
Cg

1.3106868299583232

In [17]:
a = res.scale.data
idx = np.argsort(a)

# a = a[idx]
# wt_energy = wt_energy[idx]

e_scaleint = np.trapezoid(wt_energy.data/a.reshape((a.size, 1))**2, x=a, axis=0)

factor = 1/Cg
e = np.trapezoid(e_scaleint, x=res.t.data)*factor

In [18]:
print(f'Total power  (theory): {P}')
print(f'Total energy (theory): {E}')
print(f'Total energy (computed from signal): {E_signal}')
print(f'Total energy (computed from CWT): {e}')



Total power  (theory): 0.5
Total energy (theory): 1005.25
Total energy (computed from signal): 1004.4966080678689
Total energy (computed from CWT): 618.9791256529804


### Energy is not preserved in numerical implementation of CWT (https://se.mathworks.com/matlabcentral/answers/339748-cwt-m-normalization#answer_266626)